# 1.13 Object Creation and Storage (Instantiation)

Original lesson: [1.13 Object Creation and Storage](https://pages.opencodingsociety.com/csa/unit_01/1_13)

**Key idea:** primitives (`int`, `double`, `boolean`) store their value directly on the **stack**. Objects and arrays live on the **heap**, and the stack variable only stores a **reference** (the address) to them. A method parameter is always a *copy*: a copy of the value for a primitive, and a copy of the address for an object.

## Popcorn Hack #1: Stack vs Heap

### My prediction (written before running)

- Changing `b` will **not** affect `a`. `int b = a` copies the value 10 into a separate stack slot, so `a` stays `10` and `b` becomes `20`.
- Changing `array2[0]` **will** affect `array1`. `int[] array2 = array1` copies only the reference, so both stack variables point to the same array on the heap. Both `array1[0]` and `array2[0]` print `99`.

In [1]:
public class MemoryDemo {
    public static void main(String[] args) {
        // Stack variables
        int a = 10;
        int b = a;  // Copy of value
        b = 20;     // Changing b doesn't affect a

        System.out.println("Primitives (Stack):");
        System.out.println("a = " + a);  // Still 10
        System.out.println("b = " + b);  // Now it's 20

        // Heap variables
        int[] array1 = {1, 2, 3};
        int[] array2 = array1;  // Copy of reference (address)
        array2[0] = 99;         // Changing array2 DOES affect array1

        System.out.println("\nArrays (Heap):");
        System.out.println("array1[0] = " + array1[0]);  // Now it's 99!
        System.out.println("array2[0] = " + array2[0]);  // Also 99
    }
}

MemoryDemo.main(null);

Primitives (Stack):
a = 10
b = 20

Arrays (Heap):
array1[0] = 99
array2[0] = 99

My prediction was correct.

### Tasks

**1. Why does changing `b` not affect `a`, but changing `array2` affects `array1`?**

`a` and `b` are primitives, so each one has its own slot on the stack that holds the actual number. `b = a` copied the value 10, and setting `b = 20` only overwrote `b`'s own slot. `array1` and `array2` are reference variables. `array2 = array1` copied the address of the array, not the array itself, so both stack variables point to the same array object on the heap. Writing `array2[0] = 99` changes that shared heap object, which is why `array1[0]` also reads 99.

**2. Describe what's on the stack vs. the heap for this code.**

The stack frame for `main` holds `a` (10), `b` (20), and the two references `array1` and `array2`. Both references store the same heap address. The heap holds exactly one `int[]` object, `{99, 2, 3}`. Only one array exists because `new`, or the `{1, 2, 3}` initializer, ran only once. The `args` parameter is also a reference on the stack. In this run it is `null`, so it points to nothing on the heap.

## Popcorn Hack #2: Understanding Pass-by-Reference

### My prediction (written before running)

```
Before birthday: John is 20
Inside method: John is now 21
After birthday: John is 21          <- haveBirthday changed the shared heap object

Before reassign: John is 21
Inside reassign: New Person is 99
After reassign: John is 21          <- only the local copy of the address changed
```

In [2]:
public class PersonDemo {
    static class Person {
        String name;
        int age;

        Person(String name, int age) {
            this.name = name;
            this.age = age;
        }
    }

    public static void haveBirthday(Person p) {
        p.age = p.age + 1;  // Modifying object content
        System.out.println("Inside method: " + p.name + " is now " + p.age);
    }

    public static void reassignPerson(Person p) {
        p = new Person("New Person", 99);  // Reassigning reference
        System.out.println("Inside reassign: " + p.name + " is " + p.age);
    }

    public static void main(String[] args) {
        Person john = new Person("John", 20);

        System.out.println("Before birthday: " + john.name + " is " + john.age);
        haveBirthday(john);
        System.out.println("After birthday: " + john.name + " is " + john.age);

        System.out.println("\nBefore reassign: " + john.name + " is " + john.age);
        reassignPerson(john);
        System.out.println("After reassign: " + john.name + " is " + john.age);
    }
}

PersonDemo.main(null);

Before birthday: John is 20
Inside method: John is now 21
After birthday: John is 21

Before reassign: John is 21
Inside reassign: New Person is 99
After reassign: John is 21

My prediction matched the output.

### Questions

**1. After `haveBirthday(john)` is called, what is John's age? Why?**

John is 21. The parameter `p` received a copy of `john`'s reference, so `p` and `john` point to the same `Person` object on the heap. `p.age = p.age + 1` changes that object's field, and the change is visible through `john` after the method returns.

**2. After `reassignPerson(john)` is called, what is John's name and age? Why?**

John's name is still "John" and his age is still 21. Inside `reassignPerson`, `p = new Person("New Person", 99)` created a second object on the heap and stored its address in the local parameter `p`, which is only a copy of `john`'s address. `john` on `main`'s stack never changed, so it still points to the original object. The "New Person" object has no references left after the method returns, so it can be garbage collected.

**3. Explain the difference between modifying an object's contents vs. reassigning a reference.**

Modifying contents, like `p.age = ...`, follows the reference to the heap and changes the object itself. Every variable that points to that object sees the change. Reassigning a reference, like `p = new Person(...)`, changes only which object that one stack variable points to. Because Java passes a copy of the reference into a method, reassigning the parameter never affects the caller's variable.

## Homework Hack

> The lesson's homework link (`/csa/unit_01/1_13/homework`) currently returns a 404. This homework is built from the lesson's grading plan, which asks for correct stack/heap vocabulary, predicted and explained pass-by-value vs. pass-by-reference scenarios, and objects created with constructors.

### The class: all three constructor types

`BankAccount` has a **default constructor**, a **parameterized constructor**, and a **copy constructor**. The program then runs five memory scenarios.

In [3]:
public class BankAccount {
    private String owner;
    private double balance;

    // Default constructor: no parameters, sets default values
    public BankAccount() {
        this("Unknown", 0.0);
    }

    // Parameterized constructor: caller chooses the starting values
    public BankAccount(String owner, double balance) {
        this.owner = owner;
        this.balance = balance;
    }

    // Copy constructor: a brand-new heap object with the same field values
    public BankAccount(BankAccount other) {
        this(other.owner, other.balance);
    }

    public void deposit(double amount) {
        balance += amount;
    }

    public String toString() {
        return owner + " has $" + balance;
    }
}

public class MemoryHomework {
    // Primitive parameter: receives a copy of the value
    public static void addBonusPoints(int points) {
        points += 50;
    }

    // Reference parameter: follows the copied address to the shared object
    public static void payInterest(BankAccount account) {
        account.deposit(10.0);
    }

    // Reassigning the parameter only changes the method's local copy of the address
    public static void replaceAccount(BankAccount account) {
        account = new BankAccount("Replacement", 999.0);
    }

    public static void main(String[] args) {
        int points = 100;
        addBonusPoints(points);
        System.out.println("Scenario 1 (primitive passed to a method): points = " + points);

        BankAccount mine = new BankAccount("Jacob", 100.0);
        BankAccount alias = mine;
        alias.deposit(25.0);
        System.out.println("Scenario 2 (two references, one object):    " + mine);

        BankAccount copy = new BankAccount(mine);
        copy.deposit(1000.0);
        System.out.println("Scenario 3 (copy constructor):              mine: " + mine + " | copy: " + copy);

        payInterest(mine);
        System.out.println("Scenario 4 (object contents changed):       " + mine);

        replaceAccount(mine);
        System.out.println("Scenario 5 (parameter reassigned):          " + mine);

        BankAccount blank = new BankAccount();
        System.out.println("Default constructor:                        " + blank);
        System.out.println("mine == alias? " + (mine == alias) + ", mine == copy? " + (mine == copy));
    }
}

MemoryHomework.main(null);

Scenario 1 (primitive passed to a method): points = 100
Scenario 2 (two references, one object):    Jacob has $125.0
Scenario 3 (copy constructor):              mine: Jacob has $125.0 | copy: Jacob has $1125.0
Scenario 4 (object contents changed):       Jacob has $135.0
Scenario 5 (parameter reassigned):          Jacob has $135.0
Default constructor:                        Unknown has $0.0
mine == alias? true, mine == copy? false

### Predictions (made before running) and explanations

| Scenario | Prediction | Correct? | Why (stack vs. heap) |
|-|-|-|-|
| 1. `addBonusPoints(points)` | `points = 100` | Yes | `points` is a primitive on `main`'s stack. The method got a copy of the value 100 in its own stack frame, added 50 to the copy, and discarded it when it returned. |
| 2. `alias.deposit(25)` | Jacob has $125.0 | Yes | `alias = mine` copied the **reference**, not the object. Both stack variables point to one `BankAccount` on the heap, so a deposit through `alias` shows up through `mine`. |
| 3. `copy.deposit(1000)` | mine $125.0, copy $1125.0 | Yes | The copy constructor ran `new`, which built a **second** object on the heap. `copy` points to it, so changing it leaves `mine`'s object alone. |
| 4. `payInterest(mine)` | Jacob has $135.0 | Yes | The parameter is a copy of `mine`'s address. Calling `deposit` through it changes the shared heap object's **contents**. |
| 5. `replaceAccount(mine)` | Still Jacob has $135.0 | Yes | The method reassigned its local copy of the address to a new heap object. `mine` on `main`'s stack was never touched. |
| Default constructor | Unknown has $0.0 | Yes | `new BankAccount()` calls the no-argument constructor, which chains to the parameterized one with default values. |
| `==` checks | `true`, `false` | Yes | `==` on references compares addresses. `alias` holds the same address as `mine`. `copy` holds a different object's address, even though the account data started out the same. |

**Summary:** Assigning one object variable to another copies the reference, not the object. Use `new` (here, the copy constructor) when you need an independent object.

## Self-Grade (Grading Plan, 1 point total)

| Criteria | Points | Self-grade | Evidence |
|-|-|-|-|
| Popcorn completion: predictions written before running, tasks answered in full sentences | 0.2 | 0.2 | Both Popcorn Hacks have a prediction above the cell and full-sentence answers below it. |
| Homework: stack/heap vocabulary used correctly | 0.8 (shared) | ✔ | Every explanation names where the data lives (the stack frame or the heap). |
| Homework: pass-by-value vs. pass-by-reference predicted and explained | | ✔ | Five scenarios, each with a prediction checked against real output. |
| Homework: an object created with a constructor | | ✔ | Default, parameterized, and copy constructors are all used with `new`. |
| **Total** | **1.0** | **0.95** | I left 0.05 off because the official homework page was unavailable, so I could not check this work against its exact requirements. |

**Quick validation checklist:** predictions before each Popcorn Hack ✔ · stack vs. heap identified ✔ · `new ClassName(...)` calls ✔ · object assignment always described as "copying the reference" ✔